# Modified Langevin Score Matching / Diffusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FloppingCode/modified-langevin-score-matching/blob/main/notebook.ipynb)

In [ ]:
import os, sys

# When running in Colab, clone the repo and add it to the path
if "google.colab" in sys.modules:
    if not os.path.exists("modified-langevin-score-matching"):
        !git clone https://github.com/FloppingCode/modified-langevin-score-matching.git
    sys.path.insert(0, "modified-langevin-score-matching")
else:
    sys.path.insert(0, ".")

import torch
from dsm import (
    make_dataset,
    make_dataloader,
    ScoreNetwork,
    GeometricNoiseSchedule,
    dsm_loss,
    train,
    annealed_langevin_dynamics,
    make_8gaussians_analytical_score,
    save_checkpoint,
    load_checkpoint,
)
from dsm.visualization import (
    plot_samples,
    plot_score_field,
    plot_score_comparison,
    plot_training_curves,
    plot_sampling_trajectory,
)

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = dict(
    # Dataset
    dataset="8gaussians",  # try: "moons", "swiss_roll", "circles", "8gaussians"
    n_samples=10_000,
    data_dim=2,
    # Noise schedule
    sigma_min=0.01,
    sigma_max=1.0,
    num_noise_levels=10,
    # Model
    hidden_dim=256,
    num_res_blocks=3,
    # Training
    n_epochs=200,
    batch_size=512,
    lr=1e-3,
    sigma_weighting=False,  # True: weight loss by sigma^2 (equalizes noise levels)
    # Sampling
    n_generated=2000,
    steps_per_sigma=100,
)

In [ ]:
dataset = make_dataset(CONFIG["dataset"], n_samples=CONFIG["n_samples"])
dataloader = make_dataloader(dataset, batch_size=CONFIG["batch_size"])

# Visualize
import matplotlib.pyplot as plt

data_tensor = dataset.tensors[0]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(data_tensor[:, 0], data_tensor[:, 1], s=1, alpha=0.5)
ax.set_title(f'Dataset: {CONFIG["dataset"]} ({CONFIG["n_samples"]} points)')
ax.set_aspect("equal")
plt.show()
print(f"Data shape: {data_tensor.shape}, range: [{data_tensor.min():.2f}, {data_tensor.max():.2f}]")

In [ ]:
noise_schedule = GeometricNoiseSchedule(
    sigma_min=CONFIG["sigma_min"],
    sigma_max=CONFIG["sigma_max"],
    num_levels=CONFIG["num_noise_levels"],
)

model = ScoreNetwork(
    data_dim=CONFIG["data_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    num_res_blocks=CONFIG["num_res_blocks"],
)

n_params = sum(p.numel() for p in model.parameters())
print(f"ScoreNetwork: {n_params:,} parameters")
print(f"Noise levels (σ): {noise_schedule.sigmas.tolist()}")

## Train

In [ ]:
history = train(
    model,
    dataloader,
    noise_schedule,
    n_epochs=CONFIG["n_epochs"],
    lr=CONFIG["lr"],
    device=DEVICE,
    log_every=20,
    sigma_weighting=CONFIG["sigma_weighting"],
)

In [ ]:
plot_training_curves(history)
plt.show()

## Save / Load Checkpoint

Save the trained model so you can skip retraining on future runs.

In [ ]:
# Save the trained model
CHECKPOINT_DIR = "checkpoints" if "google.colab" not in sys.modules else "modified-langevin-score-matching/checkpoints"
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{CONFIG['dataset']}_trained.pt")
save_checkpoint(model, noise_schedule, checkpoint_path, config=CONFIG, history=history)

In [ ]:
# To load instead of retraining, uncomment these lines and skip the Train section:
# model, noise_schedule, loaded_config, loaded_history = load_checkpoint(checkpoint_path, device=DEVICE)
# history = loaded_history
# print(f"Loaded model trained on '{loaded_config['dataset']}' for {loaded_config['n_epochs']} epochs")

## Score Field Visualization

Arrows should point toward data clusters at low σ and form a smooth radial field at high σ.

In [ ]:
# Plot score field at a few noise levels (largest, middle, smallest)
sigmas_to_plot = [
    noise_schedule.sigmas[0].item(),   # largest σ
    noise_schedule.sigmas[len(noise_schedule.sigmas) // 2].item(),  # middle σ
    noise_schedule.sigmas[-1].item(),  # smallest σ
]

for sigma in sigmas_to_plot:
    plot_score_field(model, sigma, device=DEVICE, data=data_tensor)
    plt.show()

## Analytical vs Learned Score (8gaussians)

For Gaussian mixtures we can compute the **exact** noised score analytically. This lets us:
1. See how well the neural network approximates the true score at each noise level
2. Compare sampling quality with perfect vs learned scores
3. Isolate whether poor samples come from bad score learning or bad sampling

In [ ]:
assert CONFIG["dataset"] == "8gaussians", "Analytical score only available for 8gaussians"

analytical_model = make_8gaussians_analytical_score().to(DEVICE)
print(f"Analytical score: {len(analytical_model.centers)} components")
print(f"  Centers at radius: {analytical_model.centers.norm(dim=-1)[0]:.4f}")
print(f"  Per-cluster sigma_data: {analytical_model.sigma_data_sq.sqrt():.4f}")

In [ ]:
# Side-by-side score fields: analytical vs learned at each noise level
for sigma in sigmas_to_plot:
    plot_score_comparison(
        analytical_model, model, sigma,
        label_a="Analytical", label_b="Learned",
        device=DEVICE, data=data_tensor,
    )
    plt.show()

In [ ]:
# Per-sigma MSE between analytical and learned scores
import numpy as np

grid_pts = np.stack(np.meshgrid(
    np.linspace(-1.5, 1.5, 30), np.linspace(-1.5, 1.5, 30)
), axis=-1).reshape(-1, 2)
grid_t = torch.tensor(grid_pts, dtype=torch.float32, device=DEVICE)

print("Per-sigma MSE (analytical vs learned):")
print("-" * 40)
with torch.no_grad():
    for sigma_val in noise_schedule.sigmas.tolist():
        sigma_t = torch.full((grid_t.shape[0], 1), sigma_val, device=DEVICE)
        s_true = analytical_model(grid_t, sigma_t)
        s_pred = model(grid_t, sigma_t)
        mse = ((s_true - s_pred) ** 2).sum(dim=-1).mean().item()
        print(f"  σ={sigma_val:.4f}  MSE={mse:.4f}")

In [ ]:
# Sample using the ANALYTICAL score (perfect oracle) for comparison
samples_analytical = annealed_langevin_dynamics(
    analytical_model,
    noise_schedule,
    n_samples=CONFIG["n_generated"],
    data_dim=CONFIG["data_dim"],
    steps_per_sigma=CONFIG["steps_per_sigma"],
    device=DEVICE,
)

# Three-panel comparison: Real | Analytical samples | Learned samples
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, pts, title in [
    (axes[0], data_tensor, "Real Data"),
    (axes[1], samples_analytical.cpu(), "Samples (Analytical Score)"),
    (axes[2], samples.cpu(), "Samples (Learned Score)"),
]:
    pts_np = pts.detach().numpy()
    ax.scatter(pts_np[:, 0], pts_np[:, 1], s=1, alpha=0.5)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

fig.suptitle("Sampling Comparison: Analytical Oracle vs Learned Model")
fig.tight_layout()
plt.show()

## Sample via Annealed Langevin Dynamics

In [ ]:
samples, trajectories = annealed_langevin_dynamics(
    model,
    noise_schedule,
    n_samples=CONFIG["n_generated"],
    data_dim=CONFIG["data_dim"],
    steps_per_sigma=CONFIG["steps_per_sigma"],
    device=DEVICE,
    return_trajectories=True,
)

print(f"Generated {samples.shape[0]} samples")

In [ ]:
plot_samples(data_tensor, samples.cpu())
plt.show()

## Sampling Trajectories

In [ ]:
plot_sampling_trajectory(trajectories, real_data=data_tensor, n_traces=50)
plt.show()